# Gold Layer Preview — Governance & Storytelling Summaries

**Project:** Music Artists & Albums Public Perception  
**Course:** Data Analysis Programming — Semester 2026-I  
**Universidad Distrital Francisco José de Caldas**

This notebook reads the **Gold Parquet files** produced by `dag_gold.py` (PySpark) and previews:
- `governance_*.parquet` — data quality KPIs with justification
- `storytelling_*.parquet` — analytical aggregations for the dashboard
- Dashboard narrative: what story the data tells

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT_DIR = os.path.abspath('..')   # one level up from notebooks/
GOLD_DIR = os.path.join(ROOT_DIR, 'datalake_gold')
NB_DIR   = os.path.abspath('.')

print('Root:     ', ROOT_DIR)
print('Gold dir: ', GOLD_DIR)

# List available Gold files
gov_files   = sorted(glob.glob(os.path.join(GOLD_DIR, 'governance_*.parquet')))
story_files = sorted(glob.glob(os.path.join(GOLD_DIR, 'storytelling_*.parquet')))
print(f'\nGovernance Parquets:   {len(gov_files)}')
print(f'Storytelling Parquets: {len(story_files)}')

---
## 1. Load Gold Parquet Files

In [ ]:
# ── Governance ─────────────────────────────────────────────────────────────
# Fallback to CSV export if no Gold Parquet is available yet
if gov_files:
    df_gov = pd.read_parquet(gov_files[-1])
    print(f'Loaded: {os.path.basename(gov_files[-1])}')
else:
    csv_path = os.path.join(NB_DIR, 'export_governance.csv')
    df_gov = pd.read_csv(csv_path)
    print(f'Gold Parquet not found — loaded fallback CSV: {csv_path}')

print(f'Shape: {df_gov.shape}')
display(df_gov.head(5))

In [ ]:
# ── Storytelling ───────────────────────────────────────────────────────────
if story_files:
    df_story = pd.read_parquet(story_files[-1])
    print(f'Loaded: {os.path.basename(story_files[-1])}')
else:
    csv_path = os.path.join(NB_DIR, 'export_storytelling.csv')
    df_story = pd.read_csv(csv_path)
    print(f'Gold Parquet not found — loaded fallback CSV: {csv_path}')

print(f'Shape: {df_story.shape}')
print('Available metric types:', df_story['metric_type'].unique().tolist())
display(df_story.head(5))

---
## 2. Governance KPIs Preview

### 2.1 Volume & Compliance Summary

In [ ]:
def pivot_kpi(df_gov, kpi_type, rename_col='value'):
    return (
        df_gov[df_gov['kpi_type'] == kpi_type][['source', 'value']]
        .set_index('source')
        .rename(columns={'value': rename_col})
    )

volume     = pivot_kpi(df_gov, 'volume',            'total_records')
compliance = pivot_kpi(df_gov, 'schema_compliance', 'schema_compliance_%')
ing_days   = pivot_kpi(df_gov, 'ingestion_days',    'ingestion_days')

summary = volume.join(compliance).join(ing_days)
summary['total_records'] = summary['total_records'].astype(int)
print('=== Gold Governance — Volume & Compliance ===')
display(summary)

### 2.2 Null Rate by Field

In [ ]:
null_gov = df_gov[df_gov['kpi_type'] == 'null_rate'].copy()
sources  = null_gov['source'].unique()

fig, axes = plt.subplots(1, len(sources), figsize=(5 * len(sources), 5))
if len(sources) == 1:
    axes = [axes]

for ax, source in zip(axes, sources):
    sub = null_gov[null_gov['source'] == source].sort_values('value', ascending=False)
    bar_colors = ['#c00000' if v > 5 else ('#ffc000' if v > 0 else '#70ad47')
                  for v in sub['value']]
    ax.barh(sub['field_name'], sub['value'], color=bar_colors, edgecolor='white')
    ax.set_xlabel('Null rate (%)')
    ax.set_title(f'{source}')
    ax.axvline(0, color='black', linewidth=0.5)

plt.suptitle('Null Rate (%) per field per source  |  🟢 0%   🟡 >0%   🔴 >5%',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### 2.3 Outlier Rate per Numeric Field

In [ ]:
outlier_gov = df_gov[df_gov['kpi_type'] == 'outlier_rate'].copy()
outlier_gov['full_field'] = outlier_gov['source'] + '.' + outlier_gov['field_name']
outlier_gov = outlier_gov.sort_values('value', ascending=True)

palette = {'reddit': '#5b9bd5', 'lastfm_artists': '#70ad47', 'lastfm_tracks': '#ffc000'}
bar_colors = [palette.get(s, '#aaa') for s in outlier_gov['source']]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(outlier_gov['full_field'], outlier_gov['value'],
        color=bar_colors, edgecolor='white')
ax.set_xlabel('Outlier rate (%)')
ax.set_title('Outlier Rate (IQR ×1.5) per numeric field')
ax.grid(axis='x', linestyle='--', alpha=0.4)

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=s) for s, c in palette.items()]
ax.legend(handles=legend_handles, fontsize=9)

plt.tight_layout()
plt.show()

### 2.4 Text Length Statistics

In [ ]:
text_kpis = ['text_len_mean', 'text_len_median', 'text_len_min', 'text_len_max']
tl = df_gov[df_gov['kpi_type'].isin(text_kpis)].copy()
tl_pivot = tl.pivot_table(
    index=['source', 'field_name'], columns='kpi_type', values='value'
).round(1)
display(tl_pivot)

### 2.5 KPI Justification Table

Each KPI is grounded in an actual observation from the silver pipeline:

In [ ]:
justifications = pd.DataFrame([
    {'KPI': 'null_rate',            'Silver Observation': '`artist`/`song` >95% unknown in Reddit',
     'Severity': 'Medium', 'Governance Action': 'Mark as optional; fill with "unknown"'},
    {'KPI': 'volume',               'Silver Observation': '228 Reddit rows; 50 artists/day in Last.fm',
     'Severity': 'Low',    'Governance Action': 'Track daily/weekly to detect ingestion gaps'},
    {'KPI': 'schema_compliance',    'Silver Observation': '100% — schema enforced before write',
     'Severity': 'N/A',   'Governance Action': 'Retain as baseline; alert if drops below 100%'},
    {'KPI': 'outlier_rate',         'Silver Observation': '`score` and `playcount` heavily right-skewed',
     'Severity': 'High',   'Governance Action': 'IQR capping applied for `score`/`word_count`; not for playcount'},
    {'KPI': 'text_len_mean/median', 'Silver Observation': 'raw_comment >> clean_comment length after NLP',
     'Severity': 'Medium', 'Governance Action': 'Confirms pipeline removes noise; monitor if ratio changes'},
    {'KPI': 'ingestion_days',       'Silver Observation': 'Last.fm: 13 snapshots in 2 days (on track)',
     'Severity': 'Low',    'Governance Action': 'Compare actual vs expected runs (1/day)'},
]).set_index('KPI')
display(justifications)

---
## 3. Storytelling Aggregations Preview

### 3.1 Sentiment Distribution

**User Story:** *"As a music label analyst, I want to know whether the community reacts positively or negatively to new releases, so I can prioritise communication campaigns."*

In [ ]:
sent = df_story[df_story['metric_type'] == 'sentiment_dist'].copy()
sent_colors = {'positive': '#70ad47', 'negative': '#c00000', 'neutral': '#5b9bd5'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bar_colors = [sent_colors.get(d, '#aaa') for d in sent['dim1']]
bars = axes[0].bar(sent['dim1'], sent['pct'], color=bar_colors,
                   edgecolor='white', width=0.5)
for bar, row in zip(bars, sent.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.5,
                 f'{row.pct:.1f}%\n(n={row.record_count})',
                 ha='center', fontsize=9)
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Sentiment Distribution (VADER)')
axes[0].set_ylim(0, 85)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Pie chart
wedge_c = [sent_colors.get(d, '#aaa') for d in sent['dim1']]
axes[1].pie(sent['record_count'], labels=sent['dim1'], colors=wedge_c,
            autopct='%1.1f%%', startangle=90, pctdistance=0.75,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Sentiment Share')

plt.suptitle('Sentiment Analysis — Reddit (r/indieheads + r/hiphopheads)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

display(sent[['dim1', 'record_count', 'pct', 'avg_score']].rename(
    columns={'dim1': 'sentiment', 'record_count': 'count', 'avg_score': 'avg_compound'}))

### 3.2 Sentiment Trend Over Time

**User Story:** *"As an artist manager, I want to monitor whether public perception improves or worsens over time to adjust the release strategy."*

In [ ]:
trend = df_story[df_story['metric_type'] == 'sentiment_trend'].copy()
trend['dim1'] = pd.to_datetime(trend['dim1'])
trend = trend.sort_values('dim1')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(trend['dim1'], trend['avg_score'], marker='o', color='#5b9bd5',
        linewidth=2, markersize=7)
ax.fill_between(trend['dim1'], 0, trend['avg_score'],
                where=trend['avg_score'] >= 0, color='#70ad47', alpha=0.2, label='Positive')
ax.fill_between(trend['dim1'], 0, trend['avg_score'],
                where=trend['avg_score'] < 0, color='#c00000', alpha=0.2, label='Negative')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Avg VADER compound score')
ax.set_xlabel('Ingestion date')
ax.set_title('Sentiment trend over time (avg compound score per ingestion date)')
ax.legend()
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

display(trend[['dim1', 'record_count', 'avg_score']].rename(
    columns={'dim1': 'date', 'record_count': 'comments', 'avg_score': 'avg_compound'}))

### 3.3 Top Keywords & Sentiment Association

**User Story:** *"As an analyst, I want to identify which terms dominate the conversation to understand what aspects of a release generate the most reaction."*

In [ ]:
kw = df_story[df_story['metric_type'] == 'top_keyword'].copy()
kw = kw.sort_values('record_count', ascending=True).tail(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Frequency (coloured by sentiment)
freq_colors = ['#70ad47' if s > 0.05 else ('#c00000' if s < -0.05 else '#5b9bd5')
               for s in kw['avg_score']]
axes[0].barh(kw['dim1'], kw['record_count'], color=freq_colors, edgecolor='white')
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top 20 keywords  (colour = avg sentiment)')
axes[0].text(0.98, 0.02, '🟢 positive  🔵 neutral  🔴 negative',
             transform=axes[0].transAxes, ha='right', fontsize=8)

# Avg sentiment per keyword
kw_s = kw.sort_values('avg_score', ascending=True)
bar_c2 = ['#70ad47' if s > 0.05 else ('#c00000' if s < -0.05 else '#5b9bd5')
          for s in kw_s['avg_score']]
axes[1].barh(kw_s['dim1'], kw_s['avg_score'], color=bar_c2, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Avg VADER compound score')
axes[1].set_title('Avg sentiment associated with each keyword')

plt.suptitle('Keyword Analysis — Reddit', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Comment Type Distribution

**User Story:** *"As an analyst, I want to distinguish recommendations from opinions to decide whether the dataset is better suited for content recommendation or sentiment analysis."*

In [ ]:
ct_dist = df_story[df_story['metric_type'] == 'comment_type_dist'].copy()
type_colors = {'recommendation': '#5b9bd5', 'opinion': '#70ad47',
               'mixed': '#ffc000', 'other': '#e07b54'}

fig, ax = plt.subplots(figsize=(8, 4))
bar_c = [type_colors.get(t, '#aaa') for t in ct_dist['dim1']]
bars = ax.barh(ct_dist['dim1'], ct_dist['pct'], color=bar_c, edgecolor='white')
for bar, row in zip(bars, ct_dist.itertuples()):
    ax.text(bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            f'{row.pct:.1f}%  (n={row.record_count})',
            va='center', fontsize=9)
ax.set_xlabel('Percentage (%)')
ax.set_title('Comment type distribution — Reddit')
ax.set_xlim(0, ct_dist['pct'].max() * 1.35)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

display(ct_dist[['dim1', 'record_count', 'pct', 'avg_score']].rename(
    columns={'dim1': 'comment_type', 'record_count': 'count', 'avg_score': 'avg_confidence'}))

### 3.5 Volume Trends by Source

**User Story:** *"As an analyst, I want to see ingestion volume over time to identify topic activity peaks and confirm ingestion regularity."*

In [ ]:
vol = df_story[df_story['metric_type'] == 'volume_trend'].copy()
vol['dim1'] = pd.to_datetime(vol['dim1'])

fig, ax = plt.subplots(figsize=(11, 4))
src_colors = {'reddit': '#5b9bd5', 'lastfm_artists': '#70ad47', 'lastfm_tracks': '#ffc000'}

for src, grp in vol.groupby('dim2'):
    grp = grp.sort_values('dim1')
    ax.plot(grp['dim1'], grp['record_count'], marker='o',
            color=src_colors.get(src, '#aaa'), label=src, linewidth=2)

ax.set_ylabel('Record count')
ax.set_xlabel('Ingestion date')
ax.set_title('Volume trend — records ingested per date per source')
ax.legend()
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.6 Top Artists & Tracks — Last.fm

**User Story:** *"As an analyst, I want the quantitative popularity ranking to compare it against qualitative Reddit perception."*

In [ ]:
top_artists = (df_story[df_story['metric_type'] == 'top_artist_lastfm']
               .sort_values('avg_score', ascending=False).head(15))
top_tracks  = (df_story[df_story['metric_type'] == 'top_track_lastfm']
               .sort_values('record_count', ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Artists by listeners
axes[0].barh(top_artists['dim1'][::-1],
             top_artists['avg_score'][::-1] / 1e6,
             color='#5b9bd5', edgecolor='white')
axes[0].set_xlabel('Unique listeners (millions)')
axes[0].set_title('Top 15 Artists — Last.fm (by unique listeners)')
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Tracks by playcount
track_labels = [
    f"{r.dim1[:22]}…\n({r.dim2})" if len(str(r.dim1)) > 22
    else f"{r.dim1}\n({r.dim2})"
    for r in top_tracks.iloc[::-1].itertuples()
]
axes[1].barh(track_labels, top_tracks['record_count'][::-1] / 1e6,
             color='#70ad47', edgecolor='white')
axes[1].set_xlabel('Plays (millions)')
axes[1].set_title('Top 15 Tracks — Last.fm (by playcount)')
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.suptitle('Quantitative Popularity Rankings — Last.fm', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 3.7 Source Comparison: API vs Scraping

**User Story:** *"As an analyst, I want to compare the data volume and nature across both sources to understand the full pipeline breadth."*

In [ ]:
vol_by_src = vol.groupby('dim2')['record_count'].sum().reset_index()
vol_by_src.columns = ['source', 'total_records']

sent_summary = (
    df_story[df_story['metric_type'] == 'sentiment_dist']
    [['dim2', 'record_count', 'avg_score']]
    .rename(columns={'dim2': 'source', 'record_count': 'comments_analysed',
                     'avg_score': 'avg_compound_score'})
)

print('=== Volume by source ===')
display(vol_by_src)

print('\n=== Sentiment by source (Reddit only) ===')
display(sent_summary)

---
## 4. Dashboard Narrative

> ### What story does the data tell?
>
> #### Core Insight: The indie and hip-hop community reacts positively to new releases — but talks about albums, not individual songs.
>
> **Finding 1 — Sentiment is predominantly positive.**  
> 67.98% of Reddit comments are positive (avg VADER score +0.62), 17.98% neutral, and only 14.04% negative. This is not superficial positivity — a compound score of +0.62 indicates genuine enthusiasm.
>
> **Finding 2 — Users react to albums as a concept, not tracks.**  
> >95% of the `artist` and `song` extraction fields are `"unknown"`, confirming that commenters discuss the release holistically rather than referencing specific songs. Campaigns should focus on album narrative, not singles.
>
> **Finding 3 — Quantitative popularity ≠ qualitative reception.**  
> The top Last.fm artists (BTS, Taylor Swift, Drake) are global chart leaders, but the Reddit communities r/indieheads and r/hiphopheads focus their discussion on niche/emerging artists. A release can generate strong organic conversation with modest streaming numbers — a key signal for A&R decisions.
>
> **Finding 4 — Data quality is high, with known limitations.**  
> 100% schema compliance across all three silver sources. The only quality issues are expected and handled: outlier capping for `score`/`word_count`, and optional-field filling for `artist`/`mbid`.
>
> **Key takeaway for the functional user:**  
> Monitor the sentiment compound score and top keywords weekly. A shift from positive keywords (`love`, `amazing`, `great`) toward negative ones (`disappointing`, `boring`, `skip`) is the earliest signal of deteriorating public reception — actionable days before it reflects in streaming numbers.

In [ ]:
# ── Executive Summary ──────────────────────────────────────────────────────
print('=' * 65)
print('EXECUTIVE SUMMARY — Gold Layer')
print('=' * 65)

sent_data = df_story[df_story['metric_type'] == 'sentiment_dist']
total_comments = int(sent_data['record_count'].sum())
pos = sent_data[sent_data['dim1'] == 'positive'].iloc[0]
neg = sent_data[sent_data['dim1'] == 'negative'].iloc[0]
neu = sent_data[sent_data['dim1'] == 'neutral'].iloc[0]

print(f'\n📊 Reddit comments analysed : {total_comments}')
print(f'   🟢 Positive : {pos["record_count"]} ({pos["pct"]:.1f}%)  avg score: +{pos["avg_score"]:.3f}')
print(f'   🔴 Negative : {neg["record_count"]} ({neg["pct"]:.1f}%)  avg score: {neg["avg_score"]:.3f}')
print(f'   🔵 Neutral  : {neu["record_count"]} ({neu["pct"]:.1f}%)  avg score: ~{neu["avg_score"]:.4f}')

top_kw = df_story[df_story['metric_type'] == 'top_keyword'].sort_values('record_count', ascending=False)
print(f'\n🔑 Top 5 keywords  : {top_kw["dim1"].head(5).tolist()}')

ta = df_story[df_story['metric_type'] == 'top_artist_lastfm'].sort_values('avg_score', ascending=False)
print(f'\n🎤 Artist #1 Last.fm: {ta.iloc[0]["dim1"]}  ({ta.iloc[0]["avg_score"]/1e6:.1f}M listeners)')

tt = df_story[df_story['metric_type'] == 'top_track_lastfm'].sort_values('record_count', ascending=False)
print(f'🎵 Track  #1 Last.fm: {tt.iloc[0]["dim1"]} — {tt.iloc[0]["dim2"]}  ({tt.iloc[0]["record_count"]/1e6:.0f}M plays)')

print(f'\n📁 Governance files  : {len(gov_files)}')
print(f'📁 Storytelling files: {len(story_files)}')
print('=' * 65)